# Libraries

In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.pardir))
import mlflow
import sentencepiece as spm
import torch
from src.model_building import load_latest_model, get_experiment, FakeNewsDetector

# Getting Model

In [2]:
model_dir = os.path.join('..','models','FakeNewsDetector')

experiment = get_experiment('FakeNewsDetector', os.path.join(model_dir, 'mlflow.db'))
mlflow.set_experiment(experiment_id = experiment.experiment_id)

<Experiment: artifact_location=('file:///d:/zPersonal/Tools/VS Code/VSCode Script '
 'Folder/GithubRepoTemps/FakeNewsDetection/notebooks/../models/FakeNewsDetector'), creation_time=1779880707986, experiment_id='1', last_update_time=1779880707986, lifecycle_stage='active', name='FakeNewsDetector', tags={}, trace_location=None, workspace='default'>

In [3]:
# Load model
model, config = load_latest_model(model_dir)
model, config

(FakeNewsDetector(
   (embed): Embedding(8000, 5, padding_idx=3)
   (conv1d): Conv1d(5, 4, kernel_size=(5,), stride=(1,))
   (pool1d): MaxPool1d(kernel_size=5, stride=5, padding=0, dilation=1, ceil_mode=False)
   (flat): Flatten(start_dim=1, end_dim=-1)
   (fc): Linear(in_features=19344, out_features=1, bias=True)
 ),
 {'vocab_size': 8000,
  'embed_dim': 5,
  'pad_id': 3,
  'conv_dim': 4,
  'kernel_size': 5,
  'max_seq': 24186})

In [4]:
# Load Tokenizer
tokenizer_dir = os.path.join('..','models','bpe')
tokenizer = spm.SentencePieceProcessor()
tokenizer.load(os.path.join(tokenizer_dir, 'spm.model'))

True

# Model Test Cases

In [5]:
# 1. Actual Credible Case (w/ Tagalog)
# 2. Fake Case
# 3. Credible but no Author
# 4: Fake Case from trainset
# 5: Actual Case but caption of a video
# 6: Actual Credible Case (w/o Tagalog)

testcase = {
    1: {
        'author': 'INQUIRER.net',
        'content': "’NAGDELIVER KAY TITO SOTTO AT ERWIN TULFO’UPDATE: Two of the 18 ‘ex-Marines’ claim they delivered suitcases of cash to Senators Vicente Sotto III and Erwin Tulfo during the Cayetano-led bloc Senate hearing.“Mabigat sa atin ‘to, dahil pinaniwalaan natin yung blue ribbon committee na pinamunuan ni Sen. Lacson, but everyone knows mag-partner talaga ‘yan, si Sen. Tito Sotto at Sen. Lacson,” Sen. Alan Peter Cayetano said.",
        'label': 0
    },
    2: {
        'author': "Adobo Chronicles",
        'content': '"Selfitis" (often misspelled as selfitie), which is a term used to describe the compulsive or obsessive-compulsive urge to take selfies and post them on social media.ResearchGate+1The Origin: From Hoax to Real StudyThe 2014 Hoax: The term originally went viral because of a spoof news article on The Adobo Chronicles claiming that the American Psychiatric Association (APA) classified selfie-taking as a new mental disorder. The APA never officially classified it as such.The Real Research: In 2017, behavioral scientists at Nottingham Trent University and the Thiagarajar School of Management decided to investigate if the phenomenon was real. They published a study confirming that selfie addiction does manifest behaviors similar to other behavioral addictions.ResearchGate+4The Three Levels of "Selfitis"Researchers mapped out three distinct stages of the behavior based on frequency and motivation:ResearchGate+2Borderline: Taking selfies at least three times a day but not posting them on social media.Acute: Taking selfies at least three times a day and posting every single one of them online.Chronic: An uncontrollable urge to take selfies around the clock, posting more than six times a day.ResearchGateWhy Does It Happen?According to insights published by Psychology Today and health researchers, the driving forces typically include:Psychology Today+1Seeking to boost low self-esteem or seeking social validation.Environmental modification (taking photos to feel belong or remember a location).Intimacy gaps and trying to gain popularity through digital feedback (likes and comments)',
        'label': 1
    },
    3: {
        'content': "IRAN CLOSES STRAIT OF HORMUZ ANEWOil prices climbed more than $2 a barrel Thursday (June 11) as Iran declared the critical energy chokepoint, the Strait of Hormuz, closed after the U.S. launched additional strikes against Iran.Iran's top joint military command announced the closure of the Strait of Hormuz on Thursday, including oil tankers and commercial ships, saying any vessel that will attempt passage will be shot at.However, the U.S. military said on X on Wednesday (June 10) that commercial ships continue to transit in and out of the strait.It also said no U.S. warships have been struck in the strait, after Iran's state media reported U.S. ships near the waterway were targeted by missiles and drones. | via Reuters",
        'label': 0
    },
    4: {
        'author':'Adobo Chronicles',
        'content':'"People Magazine is known for its international list of top celebrities and professions — from the ‘Most Beautiful Women,’ to ‘Sexiest Doctor Alive’ to “Most Influential Men and Women on Earth.’In its upcoming, special double-issue, the magazine has named the ‘Sexiest Male Models Alive.’ That’s right — models — not one or two, but three.And the winners are: U.S. President Barack Obama, Canadian Prime Minister Justin Trudeau and Mexican President Enrique Peña Nieto.In naming the trio, the magazine editors said, “Who could disagree that these men exude the look, the charisma and the bearing of fashion icons regardless of what they’re wearing — a suit, short shorts or the Filipino barong tagalog?”The cover shows the heads of state strutting their stuff at a red carpet fashion show in downtown Ottawa where they are meeting for the North American Summit, also known as the Three Amigos Summit.The special issue hits the newsstands on July 4th."',
        'label': 1
    },
    5: {
        'author': 'GMA News TV',
        'content':"WATCH: The ground can be seen shaking due to a magnitude 7.8 earthquake at Mahayahay Elementary School in Davao Occidental while a flag ceremony was being held on the first day of classes. Courtesy: DepEd Mahayahay Elementary School",
        'label': 0
    },
    6: {
        'author': 'INQUIRER.net',
        'content': "'PLEASE RESPECT MY OWN STORY' LOOK: Senator Pia Cayetano turns emotional as she responds to Senator Risa Hontiveros' speech on Wednesday, May 20. Cayetano said the trauma caused by the recent shooting incident inside the Senate continues to affect senators and their staff, as she lamented that some minority senators, whom she considered friends, allegedly failed to check on her and others after the incident. | 📸: Niño Jesus Orbeta, Philippine Daily Inquirer",
        'label': 0
    }
}

In [13]:
def model_predict(
    inputs: dict, tokenizer: spm.SentencePieceProcessor, model: FakeNewsDetector,
    max_seq: int
) -> torch.Tensor:
    """Use a model to predict the probability that a given Philippine author and news content is likely to be fake.

    Args:
        inputs (dict): A dictionary containing an 'author' and 'content'.
        tokenizer (spm.SentencePieceProcessor): Tokenizer used to encode text to numerical tokens.
        model (FakeNewsDetector): A trained FakeNewsDetector model that is used for the raw logits prediction.
        max_seq (int): Max sequence length to pad the content.

    Returns:
        torch.Tensor: Tensor that contains the prediction value for the given author and content inputs.
    """
    # Get Model Configs
    pad_id = model.embed.padding_idx
    
    # Format
    formatted = f'{inputs.get('author','Unknown')}: {inputs.get('content')}'
    encoded = torch.tensor([1] + tokenizer.encode(formatted))
    batched = encoded.unsqueeze(0)
    
    # Pad 
    pad_tensor = torch.tensor([pad_id for i in range(max_seq - batched.size(1))])
    pad_tensor = pad_tensor.unsqueeze(0)
    padded = torch.concat([
        batched,
        pad_tensor
    ], dim = -1)
    
    # Prediction
    with torch.inference_mode():
        logits = model(padded)
        prediction = 1 if logits.item() > 0 else 0
    
    return prediction

### Testcase Results

In [27]:
for idx in testcase.keys():
    pred = model_predict(
        testcase[idx],
        tokenizer,
        model,
        config['max_seq']
    )

    print(f'Testcase Idx: {idx}: Predicted Label: {pred} | Actual Label: {testcase[idx].get('label')}')

Testcase Idx: 1: Predicted Label: 1 | Actual Label: 0
Testcase Idx: 2: Predicted Label: 1 | Actual Label: 1
Testcase Idx: 3: Predicted Label: 0 | Actual Label: 0
Testcase Idx: 4: Predicted Label: 1 | Actual Label: 1
Testcase Idx: 5: Predicted Label: 0 | Actual Label: 0
Testcase Idx: 6: Predicted Label: 1 | Actual Label: 0


The model, albeit quite accurate with majority of the testcases, seems to have trouble when names are provided, specifically names of today's senates and/or officials. The presence of tagalog words in the content may also make the model suspicious of the content's veracity and truthfulness, which encourages the model to predict the content as fake when it is not.

# PyQT tests

In [2]:
from PyQt6.QtWidgets import QMainWindow, QWidget, QApplication

In [3]:
class MainWindow(QMainWindow):
    def __init__(self):
        super().__init__()
        
        # Setup Parent Window
        window = QWidget()
        self.setCentralWidget(window)

In [ ]:
%gui qt6 # Enable interactive event loop in notebook
app = QApplication([])

window = MainWindow()
window.show()

ERROR:root:Invalid GUI request 'qt6 # Enable interactive event loop in notebook', valid ones are:dict_keys(['inline', 'nbagg', 'webagg', 'notebook', 'ipympl', 'widget', None, 'qt', 'qt5', 'qt6', 'wx', 'tk', 'gtk', 'gtk3', 'osx', 'macosx', 'asyncio'])


: 

Window opens but freezes in notebook, this may be due to the test being inside a notebook environment, making it quite difficult.